# Invoice extraction: training and measuring own models

This notebook trains both models and measures them, step by step:

1. **Ground truth without manual labelling** — payment QR codes and archive file names
2. **Benchmark** — today's app: rules, alone and with Gemma 4 filling the gaps
3. **Supplier split** — train / validation / test with no supplier in two sets
4. **Own OCR** — a CRNN text recognizer **trained here from scratch**, tested against PaddleOCR
5. **Own extraction model** — LiLT **fine-tuned here** on real + generated invoices, tested on real ones
6. **Own stack only** — own OCR + rules + LiLT, with no PaddleOCR and no Gemma
7. **Every field** — supplier, IČO, VAT number, invoice number, dates, amounts — on generated invoices
8. **Headline numbers**

**How to run it.** Kernel `.venv-train` (top right), then *Run All*. With the
settings below the whole notebook takes **about 2½ hours** on the laptop GPU
(plugged in — on battery the GPU is throttled). Training shows live charts.
`TRAINING = "quick"` runs the same steps in ~15 minutes with tiny settings, to
check that everything works. Steps that need the app's OCR or Gemma run in the
app's environment (`.venv`); Gemma needs Ollama running — without it those
steps are skipped with a note and everything else still runs.

**Privacy.** Invoices stay in gitignored folders. Every cell shows aggregates
only — counts, rates, timings, training curves — never invoice contents.

In [ ]:
TRAINING = "full"  # "full": the real training (~2.5 h for the whole notebook) | "quick": smoke test (~15 min)

RUN = {
    # data preparation - already done; switch on to rebuild from scratch
    "labels": False,          # 1.  QR + file-name ground truth                         ~5 min
    "benchmark": False,       # 2.  today's app (rules, rules + Gemma) on all documents ~60-90 min
    "split": False,           # 3.  supplier-grouped split                              seconds
    "real_lines": False,      # 4a. real text lines cut from digital invoices           ~5 min
    "dump_words": False,      # 5a. words + positions of every document (PaddleOCR)     ~30-40 min
    # training and testing
    "train_ocr": True,        # 4b. train the CRNN from scratch                         ~40 min
    "test_ocr": True,         # 4c. CRNN on test-split lines                            ~1 min
    "paddle_ocr": True,       # 4d. PaddleOCR's recognizer on the same lines            ~2 min
    "page_check": True,       # 4e. own OCR on whole digital test pages                 ~1 min
    "synthetic": True,        # 5b. generate 300 fictional received invoices            ~4 min
    "synthetic_words": True,  # 5c. their words + positions                             ~6 min
    "train_lilt": True,       # 5d. fine-tune LiLT on real + generated invoices         ~30 min
    "test_lilt": True,        # 5e. LiLT vs rules vs Gemma on the real test split       ~2 min
    "gemma_vs": True,         # 5f. fair test: Gemma also asked for the variable symbol ~15 min
    "own_eval": True,         # 6a. rules on own-OCR text, real test split              ~6 min
    "own_words": True,        # 6b. own-OCR words + positions, real test split          ~5 min
    "own_lilt": True,         # 6c. LiLT on own-OCR words                               ~1 min
    "fields_data": True,      # 7a. 100 more generated invoices, every answer known     ~2 min
    "fields_paddle": True,    # 7b. app: PaddleOCR -> rules (+ Gemma), every field      ~15 min
    "fields_own": True,       # 7c. own OCR -> rules, every field                       ~4 min
}

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "extractor.py").exists())
sys.path.insert(0, str(ROOT))
from evaluation.metrics import FIELD_LABELS  # noqa: E402

SETTINGS = {
    "full": {"ocr_steps": 10000, "ocr_eval_every": 1000, "ocr_log_every": 200, "lilt_epochs": 8,
             "synthetic_train": 300, "synthetic_fields": 100, "limit": []},
    "quick": {"ocr_steps": 300, "ocr_eval_every": 100, "ocr_log_every": 25, "lilt_epochs": 1,
              "synthetic_train": 20, "synthetic_fields": 10, "limit": ["--limit", 8]},
}[TRAINING]

EXE = Path("Scripts", "python.exe") if os.name == "nt" else Path("bin", "python")
APP_PY = ROOT / ".venv" / EXE  # the app's environment: PaddleOCR, Ollama client
TRAIN_PY = ROOT / ".venv-train" / EXE  # PyTorch + Transformers
LLM = "gemma4:e4b-it-qat"

# shared data (prepared once)
DOCS = ROOT / "eval_data" / "docs"
LABELS = ROOT / "eval_data" / "labels_qr.csv"
LABELS_TEST = ROOT / "eval_data" / "labels_qr_test.csv"  # the same labels, test split only
BENCHMARK = ROOT / "eval_results" / "qr_labels_gemma4"
SPLIT = ROOT / "training" / "data" / "split.csv"
REAL_LINES = ROOT / "training" / "data" / "real_lines"
WORDS = ROOT / "training" / "data" / "words"

# everything this notebook trains or measures goes here - never over the earlier results
OUT = ROOT / "training" / "runs" / ("notebook" if TRAINING == "full" else "notebook_quick")
OCR_RUN = OUT / "crnn"
LILT_RUN = OUT / "lilt"
SYNTH = OUT / "synthetic_train"  # generated invoices LiLT trains on
FIELDS_SET = OUT / "synthetic_fields"  # other generated invoices, for scoring every field
PCT = "{:.1%}"


def run(python, *args, on_line=None, optional=False):
    # Run one pipeline step as a module in the given environment, streaming its progress.
    # on_line(line) -> True consumes a line (a live chart); optional steps warn instead of stopping Run All.
    cmd = [str(python), "-m", *map(str, args)]
    print(">", " ".join(cmd[2:]), flush=True)
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    with subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as process:
        for line in process.stdout:
            if "Redirects are currently not supported" in line or (on_line and on_line(line)):
                continue
            print(line, end="", flush=True)
    if process.returncode:
        if optional:
            print(f"!! step skipped (exit code {process.returncode}) - the rest of the notebook continues")
            return False
        raise RuntimeError(f"step failed with exit code {process.returncode}")
    return True


class LiveChart:
    # Redraws training curves in place from the JSON progress lines a training script prints.
    def __init__(self, x, panels):
        self.x, self.panels, self.rows = x, panels, []  # panels: [(title, ylabel, [columns]), ...]
        self.handle = display(display_id=True)

    def __call__(self, line):
        try:
            row = json.loads(line)
        except ValueError:
            return False
        if not isinstance(row, dict) or self.x not in row:
            return False
        self.rows.append(row)
        self.draw()
        return True

    def draw(self):
        fig, axes = plt.subplots(1, len(self.panels), figsize=(14, 3.5))
        for ax, (title, ylabel, columns) in zip(axes, self.panels):
            for column in columns:
                points = [(r[self.x], r[column]) for r in self.rows if r.get(column) is not None]
                if points:
                    ax.plot(*zip(*points), marker="o", markersize=3, label=column)
            ax.set(title=title, xlabel=self.x, ylabel=ylabel)
            if ax.get_legend_handles_labels()[0]:
                ax.legend(fontsize=8)
        fig.suptitle(f"training now - {len(self.rows)} progress reports", fontsize=10)
        plt.tight_layout()
        self.handle.update(fig)
        plt.close(fig)


def load(path):
    path = Path(path)
    if not path.exists():
        print(f"not produced yet: {path.relative_to(ROOT)}")
        return None
    return json.loads(path.read_text(encoding="utf-8"))


def read_csv(path):
    path = Path(path)
    if not path.exists():
        print(f"not produced yet: {path.relative_to(ROOT)}")
        return None
    return pd.read_csv(path, encoding="utf-8-sig", dtype=str).fillna("")


def rates(correct, wrong, missing):
    total, filled = correct + wrong + missing, correct + wrong
    accuracy = correct / total if total else None
    precision = correct / filled if filled else None
    if precision is None:
        return accuracy, precision, None
    return accuracy, precision, (2 * precision * accuracy / (precision + accuracy) if precision + accuracy else 0.0)


def score_rows(name, scores):
    rows = []
    for field, s in scores.items():
        total = s["correct"] + s["wrong"] + s["missing"]
        if total:
            acc, prec, f1 = rates(s["correct"], s["wrong"], s["missing"])
            rows.append({"setup": name, "field": FIELD_LABELS.get(field, field), "invoices": total,
                         "correct": s["correct"], "wrong": s["wrong"], "missing": s["missing"],
                         "accuracy (= recall)": acc, "precision": prec, "F1": f1})
    return rows


def pooled(scores):
    c, w, m = (sum(s[k] for s in scores.values()) for k in ("correct", "wrong", "missing"))
    return rates(c, w, m)


def show_scores(table):
    display(table.style.format({c: PCT for c in ["accuracy (= recall)", "precision", "F1"]}, na_rep="–")
            .hide(axis="index"))


def bars(table, value, title, ax):
    table.pivot_table(index="field", columns="setup", values=value, sort=False).plot.barh(ax=ax, xlim=(0, 1))
    ax.set(title=title, xlabel=value, ylabel="")
    ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
    ax.legend(loc="lower left", fontsize=8)


def compare(setups, title):
    # setups: {name: per-field scores}; table + accuracy / precision charts + pooled line per setup
    table = pd.DataFrame([r for name, scores in setups.items() for r in score_rows(name, scores)])
    if table.empty:
        return
    print(title)
    show_scores(table)
    fig, axes = plt.subplots(1, 2, figsize=(14, max(3.0, 0.5 * table["field"].nunique() * len(setups) / 2)))
    bars(table, "accuracy (= recall)", "accuracy", axes[0])
    bars(table, "precision", "precision", axes[1])
    plt.tight_layout()
    plt.show()
    for name, scores in setups.items():
        acc, prec, f1 = pooled(scores)
        print(f"  {name}: all fields pooled - accuracy {acc:.1%}, precision {prec:.1%}, F1 {f1:.1%}")


def run_scores(summary_path):
    # {setup name: per-field scores} from an evaluation run's summary.json
    summary = load(summary_path)
    return {s["name"]: s["scores"] for s in summary["setups"] if "scores" in s} if summary else {}


def lilt_scores(summary_path, group="all"):
    # {setup name: per-field scores} for one group of a LiLT test report
    summary = load(summary_path)
    return {name: s["fields"] for name, s in summary.get(group, {}).items()} if summary else {}


OUT.mkdir(parents=True, exist_ok=True)
print("repository:", ROOT, "| mode:", TRAINING, "| results go to", OUT.relative_to(ROOT))
print("app python:", APP_PY.exists(), "| training python:", TRAIN_PY.exists())

## 1. Ground truth without manual labelling

Most Slovak invoices carry a **PAY by square** QR code, Czech ones a **QR Platba**
code. The issuer's accounting system generates them from the invoice data and
the extractor never reads them, so they are an independent answer key for the
**amount to pay and the variable symbol**. The archive file name adds the
**direction** (received invoices are filed under the internal number assigned on
approval) and, for issued invoices, the **invoice number**.

Safeguards: PAY by square payloads are CRC-checked; documents with two different
payment codes, or an issued invoice whose QR variable symbol differs from its
file name, get no labels from that source. The QR code's **date is not used**: on
220 of 242 issued invoices it is the issue date, not the due date.

Other fields (supplier, IČO, VAT number, dates, net and VAT amounts) have no
independent answer on real invoices without the office's approved records —
section 7 scores them on generated invoices instead.

In [ ]:
if RUN["labels"]:
    run(APP_PY, "evaluation.payment_qr", "--pdfs", DOCS, "--out", LABELS)

labels = read_csv(LABELS)
if labels is not None:
    columns = {"typ": "Direction", "cislo_faktury": "Invoice number", "suma_s_dph": "Amount to pay",
               "variabilny_symbol": "Variable symbol"}
    coverage = pd.DataFrame([{"field": name, "documents with a known value": int((labels[c] != "").sum())}
                             for c, name in columns.items() if c in labels.columns])
    print(f"{len(labels)} labelled documents")
    display(coverage.style.hide(axis="index"))

## 2. Benchmark: today's app

The production extractor on every document: the PDF text layer or PaddleOCR
(PP-OCRv6, GPU) for scans → rules with arithmetic validation → for received
invoices with amounts or dates still empty, Gemma 4 (local, via Ollama) fills
the gaps. *Accuracy (= recall)*: share of invoices with the right value; a wrong
and an empty value both count as misses. *Precision*: share of filled-in values
that are right — the one that matters most, because a wrong value can slip past
review.

In [ ]:
if RUN["benchmark"]:
    run(APP_PY, "evaluation", "--pdfs", DOCS, "--labels", LABELS, "--llm", LLM, "--out", BENCHMARK)

summary = load(BENCHMARK / "summary.json")
if summary:
    print(f"{summary['documents']} documents: {summary['text_layer_docs']} digital, {summary['ocr_docs']} scanned")
    compare(run_scores(BENCHMARK / "summary.json"), "All labelled documents")

## 3. Supplier split

A model that has seen a supplier's layout in training would look better on that
supplier's test invoices than on a new supplier's. Documents are grouped by
counterparty first — the supplier's IBAN from the QR code for received invoices,
else the IČO or name — and whole groups go to train (70 %), validation (10 %) or
test (20 %).

In [ ]:
if RUN["split"]:
    run(APP_PY, "training.split", "--labels", LABELS, "--predictions", BENCHMARK / "per_document.csv",
        "--out", SPLIT)

split = read_csv(SPLIT)
if split is not None:
    table = split.assign(kind=split["typ"].replace({"Prijatá": "received", "Vystavená": "issued", "": "unknown"}))
    counts = table.pivot_table(index="split", columns="kind", values="file", aggfunc="count", fill_value=0)
    counts["supplier groups"] = table.groupby("split")["group"].nunique()
    display(counts.loc[[s for s in ["train", "val", "test"] if s in counts.index]])
    if labels is not None:  # the test split's labels, for the steps that score only test invoices
        labels[labels["file"].isin(set(split.loc[split["split"] == "test", "file"]))].to_csv(
            LABELS_TEST, index=False, encoding="utf-8-sig")

## 4. Own OCR: a CRNN text recognizer, trained from scratch

No pretrained weights. A CNN reads the line image, a two-layer bidirectional LSTM
reads the features left to right, CTC loss aligns them with the text (8.5 M
parameters). Training lines are **synthetic** — invoice field patterns rendered
with 30 system fonts and degraded like a scan — mixed 70 / 30 with **real lines
cut from digital invoices** of training suppliers, whose text layer gives an exact
transcription. Below: synthetic samples (generated text, not invoice data).

In [ ]:
import random

from training.ocr import synth

fonts = synth.available_fonts()
sampler = synth.TextSampler()
fig, axes = plt.subplots(8, 1, figsize=(10, 6))
for i, ax in enumerate(axes):
    rng = random.Random(1000 + i)
    text = sampler.sample(rng)
    ax.imshow(synth.render_line(text, rng.choice(fonts), rng), cmap="gray", vmin=0, vmax=255)
    ax.set_title(text, fontsize=9, loc="left")
    ax.axis("off")
plt.tight_layout()
plt.show()

if RUN["real_lines"]:
    run(TRAIN_PY, "training.ocr.real_lines", "--pdfs", DOCS, "--split", SPLIT, "--out", REAL_LINES)
lines = read_csv(REAL_LINES / "lines.csv")
if lines is not None:
    display(lines.groupby("split").agg(lines=("text", "size"), documents=("document", "nunique")))

In [ ]:
if RUN["train_ocr"]:
    run(TRAIN_PY, "training.ocr.train", "--real", REAL_LINES, "--out", OCR_RUN,
        "--steps", SETTINGS["ocr_steps"], "--eval-every", SETTINGS["ocr_eval_every"],
        "--log-every", SETTINGS["ocr_log_every"],
        on_line=LiveChart("step", [("Training loss (CTC)", "loss", ["loss"]),
                                   ("Validation CER (lower is better)", "CER", ["real_val_cer", "synthetic_val_cer"])]))

history = load(OCR_RUN / "history.json")
if history:
    h = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for column in [c for c in h.columns if c.endswith("_cer")]:
        axes[0].plot(h["step"], h[column] * 100, marker="o", label=column.removesuffix("_cer"))
    axes[0].set(title="Character error rate on validation lines (lower is better)", xlabel="step", ylabel="CER %")
    axes[0].legend()
    axes[1].plot(h["step"], h["loss"], marker="o")
    axes[1].set(title="Training loss (CTC)", xlabel="step")
    plt.tight_layout()
    plt.show()
    display(h.tail(3).style.hide(axis="index"))

**Test.** Lines from test-split suppliers, never seen in training. PaddleOCR's
recognizer (PP-OCRv6, the model the app uses) is scored on exactly the same
lines. Metrics are also shown with spaces ignored, because a PDF text layer and
an OCR engine space words differently — that alone is not a misreading.

**Whole pages.** The CRNN reads one line at a time; `training/ocr/detect.py` finds
the lines with classical image processing (no learned model): local threshold,
straightening, erasing table rules, joining letters into phrases. The page check
reads whole rendered digital test pages and compares them with the text layer.

In [ ]:
if RUN["test_ocr"]:
    run(TRAIN_PY, "training.ocr.train", "--real", REAL_LINES, "--test", OCR_RUN / "best.pt")
if RUN["paddle_ocr"]:
    run(APP_PY, "training.ocr.paddle_baseline", "--real", REAL_LINES, "--out", OUT / "paddle_test.json",
        optional=True)
if RUN["page_check"]:
    run(TRAIN_PY, "training.ocr.page_check", "--pdfs", DOCS, "--split", SPLIT, "--which", "test",
        "--model", OCR_RUN / "best.pt", "--out", OUT / "page_check_test.json")

ocr_test, paddle_test = load(OCR_RUN / "test_metrics.json"), load(OUT / "paddle_test.json")
columns = {"cer": "CER", "cer_no_spaces": "CER, spaces ignored", "line_accuracy": "lines exact",
           "line_accuracy_no_spaces": "lines exact, spaces ignored", "lines": "lines"}
ocr_rows = {}
if ocr_test:
    ocr_rows["Own CRNN - real test lines"] = ocr_test.get("real_test")
    ocr_rows["Own CRNN - synthetic test lines"] = ocr_test.get("synthetic_test")
if paddle_test:
    ocr_rows["PaddleOCR PP-OCRv6 - real test lines"] = paddle_test.get("real_test")
ocr_rows = {k: {columns[c]: v[c] for c in columns if c in v} for k, v in ocr_rows.items() if v}
if ocr_rows:
    display(pd.DataFrame(ocr_rows).T.style.format({c: PCT for c in columns.values() if c != "lines"}, na_rep="–"))

page_check = load(OUT / "page_check_test.json")
if page_check:
    display(pd.DataFrame([{
        "digital test pages": page_check["pages"], "text-layer words": page_check["text_layer_words"],
        "words inside a detected phrase": page_check["words_inside_a_segment"],
        "words read exactly": page_check["words_read_exactly"],
        "CER, spaces ignored": page_check["cer_no_spaces"],
        "seconds per page": page_check["seconds_per_page"]}]).style.format(
        {"words inside a detected phrase": PCT, "words read exactly": PCT, "CER, spaces ignored": "{:.2%}",
         "seconds per page": "{:.2f}"}).hide(axis="index"))

## 5. Own extraction model: LiLT, fine-tuned on real + generated invoices

**LiLT** (`nielsr/lilt-xlm-roberta-base`, MIT licence) reads every word together
with its position on the page. It is fine-tuned to tag the **amount to pay** and
the **variable symbol** — the fields the QR codes give an answer for. Training
labels are automatic: words that read as the QR values are tagged (BIO scheme).

Only 149 real received invoices are in the training split, so **300 generated
received invoices** are added (`synthetic_invoices.py`: fictional suppliers, many
fonts, wordings in Slovak, Czech and English, number formats, layouts, deposits
that make the amount to pay differ from the total — half digital, half scans).
They are training data only: the test below is on **real** invoices from
suppliers the model never saw.

In [ ]:
if RUN["dump_words"]:
    run(APP_PY, "training.extraction.dump_words", "--pdfs", DOCS, "--out", WORDS)
if RUN["synthetic"]:
    run(TRAIN_PY, "training.extraction.synthetic_invoices", "--count", SETTINGS["synthetic_train"],
        "--seed", 11, "--out", SYNTH)
if RUN["synthetic_words"]:
    run(APP_PY, "training.extraction.dump_words", "--pdfs", SYNTH / "pdfs", "--out", SYNTH / "words")

pdfs = sorted((SYNTH / "pdfs").glob("*.pdf"))
if pdfs:  # four generated invoices (fictional companies)
    import pypdfium2 as pdfium

    fig, axes = plt.subplots(1, 4, figsize=(16, 5.5))
    for ax, path in zip(axes, pdfs[:4]):
        pdf = pdfium.PdfDocument(str(path))
        ax.imshow(pdf[0].render(scale=0.8).to_pil().convert("L"), cmap="gray")
        pdf.close()
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
if RUN["train_lilt"]:
    run(TRAIN_PY, "training.extraction.train_lilt", "--words", WORDS, "--labels", LABELS, "--split", SPLIT,
        "--synthetic", SYNTH, "--out", LILT_RUN, "--epochs", SETTINGS["lilt_epochs"],
        on_line=LiveChart("epoch", [("Training loss", "loss", ["loss"]),
                                    ("Validation accuracy (real invoices)", "accuracy",
                                     ["val_accuracy_all_fields", "val_acc_suma_s_dph", "val_acc_variabilny_symbol"])]))

config = load(LILT_RUN / "config.json")
if config:
    print({k: config.get(k) for k in ["train_documents", "synthetic_train_documents",
                                      "train_documents_skipped_unmatched", "val_documents", "train_chunks", "epochs"]})
history = load(LILT_RUN / "history.json")
if history:
    h = pd.DataFrame(history)
    fig, ax = plt.subplots(figsize=(8, 4))
    for column in [c for c in h.columns if c.startswith("val_acc")]:
        ax.plot(h["epoch"], h[column], marker="o", label=column.removeprefix("val_acc_"))
    ax.set(title="Validation accuracy per epoch (real invoices)", xlabel="epoch", ylim=(0, 1.02))
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
    ax.legend()
    plt.show()

**Test on the real test split.** Today's app (rules, rules + Gemma 4), LiLT alone,
and the design the app would use: **rules first, LiLT only for fields the rules
left empty**. The app asks Gemma only for amounts and dates, never for the variable
symbol, so a second, fair run also asks Gemma for the variable symbol.

In [ ]:
if RUN["test_lilt"]:
    run(TRAIN_PY, "training.extraction.train_lilt", "--words", WORDS, "--labels", LABELS, "--split", SPLIT,
        "--test", LILT_RUN / "best", "--baseline", BENCHMARK / "per_document.csv")
if RUN["gemma_vs"] and run(APP_PY, "evaluation", "--pdfs", DOCS, "--labels", LABELS_TEST, "--llm", LLM,
                           "--llm-also", "variabilny_symbol", "--out", OUT / "gemma_vs", *SETTINGS["limit"],
                           optional=True):
    run(TRAIN_PY, "training.extraction.train_lilt", "--words", WORDS, "--labels", LABELS, "--split", SPLIT,
        "--test", LILT_RUN / "best", "--baseline", OUT / "gemma_vs" / "per_document.csv",
        "--report-dir", OUT / "lilt_vs_fair_gemma")

GEMMA = f"Rules: Regex + LLM fill-in ({LLM})"
GEMMA_VS = f"Rules: Regex + LLM fill-in ({LLM}, also variabilny_symbol)"
for group, title in [("all", "All real test invoices"), ("Prijatá", "Received"), ("scanned", "Scanned")]:
    main, fair = lilt_scores(LILT_RUN / "test_summary.json", group), lilt_scores(
        OUT / "lilt_vs_fair_gemma" / "test_summary.json", group)
    setups = {"Rules (today)": main.get("Rules: Regex only"),
              "Rules + Gemma 4 (today's app)": main.get(GEMMA),
              "Rules + Gemma 4, also asked for VS (fair)": fair.get(GEMMA_VS),
              "LiLT alone": main.get("LiLT (own model)"),
              "Rules -> LiLT for empty fields": main.get("Rules: Regex only -> LiLT for gaps")}
    setups = {k: v for k, v in setups.items() if v}
    if setups:
        compare(setups, f"\n{title}")

## 6. Own stack only: own OCR + rules + LiLT — no PaddleOCR, no Gemma

Scanned test invoices read by the own OCR (classical line detection + the CRNN
trained above), the rules on that text, and LiLT on the own OCR's word boxes for
fields the rules left empty. Digital invoices use their text layer, as in the app.

In [ ]:
if RUN["own_eval"]:
    run(TRAIN_PY, "evaluation", "--pdfs", DOCS, "--labels", LABELS_TEST, "--ocr", "own",
        "--ocr-model", OCR_RUN / "best.pt", "--out", OUT / "own_eval", *SETTINGS["limit"])
if RUN["own_words"]:
    run(TRAIN_PY, "training.extraction.dump_words", "--pdfs", DOCS, "--out", OUT / "words_own", "--ocr", "own",
        "--ocr-model", OCR_RUN / "best.pt", "--split", SPLIT, "--which", "test")
if RUN["own_lilt"]:
    run(TRAIN_PY, "training.extraction.train_lilt", "--words", OUT / "words_own", "--labels", LABELS,
        "--split", SPLIT, "--test", LILT_RUN / "best", "--baseline", OUT / "own_eval" / "per_document.csv",
        "--report-dir", OUT / "lilt_own_ocr")

for group, title in [("all", "All real test invoices"), ("scanned", "Scanned")]:
    paddle, own = lilt_scores(LILT_RUN / "test_summary.json", group), lilt_scores(
        OUT / "lilt_own_ocr" / "test_summary.json", group)
    setups = {"Today's app: PaddleOCR -> rules -> Gemma 4": paddle.get(GEMMA),
              "PaddleOCR -> rules -> LiLT": paddle.get("Rules: Regex only -> LiLT for gaps"),
              "Own stack: own OCR -> rules -> LiLT": own.get("Rules: Regex only -> LiLT for gaps"),
              "Own OCR -> LiLT alone": own.get("LiLT (own model)")}
    setups = {k: v for k, v in setups.items() if v}
    if setups:
        compare(setups, f"\n{title}")
own_summary = load(OUT / "own_eval" / "summary.json")
if own_summary and summary:
    speed = pd.DataFrame([{"stage": label, **t} for label, t in summary["timings"][1:2] + own_summary["timings"][1:2]])
    display(speed.style.format({c: "{:.2f} s" for c in ["median", "mean", "p90", "max"]}).hide(axis="index"))

## 7. Every field, on generated invoices

The QR codes only answer for the amount to pay and the variable symbol. For the
other fields — **supplier name, IČO, VAT number (IČ DPH), invoice number, issue,
delivery and due dates, net and VAT amounts** — a second set of generated
invoices (another seed, not the ones LiLT trained on) has every answer known.
The rules and Gemma never saw any generated invoice, so this is a fair test of
the extractor on unfamiliar suppliers. It is **not** accuracy on TDS's real
invoices: the layouts come from one generator. *Total amount* here is the amount
to pay (as in the QR code); invoices with a deposit make it differ from the total
with VAT.

In [ ]:
if RUN["fields_data"]:
    run(TRAIN_PY, "training.extraction.synthetic_invoices", "--count", SETTINGS["synthetic_fields"],
        "--seed", 23, "--out", FIELDS_SET)
if RUN["fields_paddle"]:
    run(APP_PY, "evaluation", "--pdfs", FIELDS_SET / "pdfs", "--labels", FIELDS_SET / "labels.csv",
        "--llm", LLM, "--out", OUT / "fields_paddle", optional=True)
if RUN["fields_own"]:
    run(TRAIN_PY, "evaluation", "--pdfs", FIELDS_SET / "pdfs", "--labels", FIELDS_SET / "labels.csv",
        "--ocr", "own", "--ocr-model", OCR_RUN / "best.pt", "--out", OUT / "fields_own")

fields_paddle, fields_own = run_scores(OUT / "fields_paddle" / "summary.json"), run_scores(
    OUT / "fields_own" / "summary.json")
setups = {"PaddleOCR -> rules": fields_paddle.get("Regex only"),
          "PaddleOCR -> rules + Gemma 4": fields_paddle.get(f"Regex + LLM fill-in ({LLM})"),
          "Own OCR -> rules": fields_own.get("Regex only")}
setups = {k: v for k, v in setups.items() if v}
if setups:
    n = load(OUT / "fields_own" / "summary.json") or load(OUT / "fields_paddle" / "summary.json")
    compare(setups, f"{n['documents']} generated received invoices ({n['ocr_docs']} scanned), every field")

## 8. Headline numbers

Every scored field pooled. The first rows are on the real **test split** (82
invoices from suppliers never seen in training, amount to pay + variable symbol);
the last rows on **generated** invoices, every field.

In [ ]:
rows = []
main = lilt_scores(LILT_RUN / "test_summary.json")
own = lilt_scores(OUT / "lilt_own_ocr" / "test_summary.json")
fair = lilt_scores(OUT / "lilt_vs_fair_gemma" / "test_summary.json")
for name, scores, where in [
        ("Today's app: rules + Gemma 4", main.get(GEMMA), "real test split"),
        ("Rules + Gemma 4, also asked for VS", fair.get(GEMMA_VS), "real test split"),
        ("Rules -> own LiLT (PaddleOCR text)", main.get("Rules: Regex only -> LiLT for gaps"), "real test split"),
        ("Own stack: own OCR -> rules -> own LiLT", own.get("Rules: Regex only -> LiLT for gaps"), "real test split"),
        ("App: PaddleOCR -> rules + Gemma 4", fields_paddle.get(f"Regex + LLM fill-in ({LLM})"),
         "generated, every field"),
        ("Own OCR -> rules", fields_own.get("Regex only"), "generated, every field")]:
    if scores:
        acc, prec, f1 = pooled(scores)
        rows.append({"what": name, "set": where, "accuracy": acc, "precision": prec, "F1": f1})
if rows:
    display(pd.DataFrame(rows).style.format({c: PCT for c in ["accuracy", "precision", "F1"]}, na_rep="–")
            .hide(axis="index"))
for name, m in ocr_rows.items():
    print(f"{name}: character error rate {m['CER']:.2%} ({m.get('CER, spaces ignored', float('nan')):.2%} "
          f"ignoring spaces) on {m['lines']} lines")